# Fusion multimodale imagerie + tabulaire en SSL — démo de bout en bout

Version simplifiée de **Hager et al., CVPR 2023** appliquée à la mammographie.

Ce notebook déroule les 7 étapes du projet en montrant, à chaque fois, *ce qu'on regarde pour savoir si ça marche*.

| Étape | Ce qu'on fait | Le contrôle |
|---|---|---|
| 1 | Construire les paires (image, dossier clinique) | inspecter les images et la structure des manquants |
| 2 | Prétraiter le tabulaire sans mentir sur les manquants | vérifier `x_num` / `x_cat` / `mask` |
| 3 | Deux encodeurs | formes des sorties, nombre de paramètres |
| 4 | Pré-entraînement contrastif | la perte descend sous `log(B)`, le retrieval décolle |
| 5 | Fine-tuning aval | AUROC test avec IC bootstrap |
| 6 | Comparer aux baselines | tableau + courbe d'efficacité en labels |
| 7 | Ablations | robustesse aux métadonnées manquantes |

> **Colab** : exécuter la cellule d'installation ci-dessous, puis `Exécution > Tout exécuter`.
> Sur GPU T4, le notebook complet prend ~15 min avec les réglages par défaut.


In [ ]:
# --- Colab uniquement : décommenter ---
# !pip install -q torch torchvision timm matplotlib tabulate
# !git clone https://github.com/<votre-compte>/fusion-multimodale-mammo.git
# %cd fusion-multimodale-mammo

import sys, os
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():        # si le notebook est lancé depuis notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
print("racine du projet :", ROOT)

import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())

---
## Étape 0 — Configuration

Tout passe par `configs/default.yaml`. On l'allège ici pour que le notebook tourne vite ;
les valeurs entre parenthèses sont celles à utiliser pour un vrai run.

In [ ]:
from src.utils import load_config, set_seed, get_device, count_params

FAST = True   # False -> réglages "vrai run"

cfg = load_config("configs/default.yaml", {
    "data.n_synthetic":      2000 if FAST else 8000,
    "data.image_size":       96   if FAST else 224,
    "data.num_workers":      2,
    "pretrain.epochs":       15   if FAST else 100,
    "pretrain.batch_size":   128  if FAST else 256,
    "finetune.epochs":       10   if FAST else 30,
    "model.image.name":      "resnet18" if FAST else "resnet50",
    "out_dir":               "outputs/notebook",
})
set_seed(cfg.seed)
device = get_device(cfg.device)
print("device :", device)
print({k: cfg[k] for k in ("seed", "out_dir")})

---
## Étape 1 — Les paires (image, dossier clinique)

L'unité d'apprentissage n'est pas une image : c'est une **paire**. Le seul signal utilisé pendant
le pré-entraînement est « cette image appartient à ce patient ». Aucune étiquette de malignité
n'intervient avant l'étape 5.

Le générateur synthétique fait dériver l'image **et** le tableau des mêmes variables latentes
(âge → densité → texture ; risque → type de lésion → apparence). Sans cette information partagée,
l'alignement contrastif n'aurait rien à apprendre.

In [ ]:
from src.data.synthetic import build_synthetic_dataset, NUM_COLS, CAT_COLS

images, meta, descriptions = build_synthetic_dataset(
    n=cfg.data.n_synthetic, image_size=cfg.data.image_size,
    missing_rate=cfg.data.missing_rate, seed=cfg.seed,
    cache_dir=Path(cfg.out_dir) / "cache")

print("images :", images.shape, images.dtype)
print("métadonnées :", meta.shape)
print("prévalence de la malignité :", round(meta.malignancy.mean(), 3))
meta[["age", "breast_density", "abnormality_type", "mass_margins",
      "calc_type", "subtlety", "malignancy", "birads"]].head()

In [ ]:
# Quelques exemples, bénins vs malins
fig, axes = plt.subplots(2, 6, figsize=(15, 5.4))
for row, (lab, title) in enumerate([(0, "BÉNIN"), (1, "MALIN")]):
    idx = np.where(meta.malignancy.values == lab)[0][:6]
    for ax, i in zip(axes[row], idx):
        ax.imshow(images[i], cmap="gray"); ax.axis("off")
        r = meta.iloc[i]
        desc = r.mass_margins if r.abnormality_type == "MASS" else r.calc_type
        ax.set_title(f"{r.abnormality_type[:4]} · {desc}", fontsize=7)
    axes[row, 0].set_ylabel(title)
    axes[row, 0].axis("on"); axes[row, 0].set_xticks([]); axes[row, 0].set_yticks([])
fig.suptitle("Pseudo-mammographies : la morphologie de la lésion porte le signal", y=1.0)
plt.tight_layout(); plt.show()

### La structure des valeurs manquantes

C'est le point le plus important de la figure suivante. La manquance n'est **pas aléatoire** :

* `mass_shape` / `mass_margins` sont vides pour tous les cas de **calcification** ;
* `calc_type` / `calc_distribution` sont vides pour tous les cas de **masse**.

Cette manquance *porte de l'information* (MNAR). Une imputation par la moyenne la détruit ;
c'est précisément ce que l'encodeur missing-aware évite.

In [ ]:
cols = NUM_COLS + CAT_COLS
M = meta[cols].isna().to_numpy()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 4.6),
                             gridspec_kw={"width_ratios": [2.2, 1]})
order = np.argsort(meta.abnormality_type.values, kind="stable")
a1.imshow(M[order][:400].T, aspect="auto", cmap="Greys", interpolation="nearest")
a1.set_yticks(range(len(cols))); a1.set_yticklabels(cols, fontsize=8)
a1.set_xlabel("patients (triés par type d'anomalie)")
a1.set_title("Carte des valeurs manquantes — noir = manquant")

rate = M.mean(0)
a2.barh(range(len(cols)), rate, color=["#c44" if r > .4 else "#48a" for r in rate])
a2.set_yticks(range(len(cols))); a2.set_yticklabels(cols, fontsize=8)
a2.set_xlabel("taux de manquants"); a2.set_title("Par colonne")
a2.axvline(cfg.data.missing_rate, ls="--", c="k", lw=1)
plt.tight_layout(); plt.show()

print("Les barres rouges (>40 %) sont les colonnes à manquance STRUCTURELLE :")
print("elles n'existent que pour un sous-type de lésion.")

---
## Étape 2 — Prétraitement et splits

Trois précautions décident de la validité de tout le reste :

1. **Anti-fuite** — `pathology`, `assessment`, `breast_birads` sont retirées du vecteur tabulaire.
2. **Statistiques ajustées sur le train seulement** — moyennes, écarts-types, vocabulaires.
3. **Split par patient** quand un identifiant est disponible (sur les vrais datasets).

In [ ]:
from src.data.datamodule import build_datasets

bundle = build_datasets(cfg, seed=cfg.seed)
s = bundle.schema
print(f"train={len(bundle.train_ds)}  val={len(bundle.val_ds)}  test={len(bundle.test_ds)}")
print(f"features tabulaires : {s.n_features}  ({s.n_num} numériques + {s.n_cat} catégorielles)")
print(f"classes : {bundle.n_classes} | poids de classe : {bundle.class_weights.round(3)}")
print(f"taux global de manquants (train) : {bundle.train_ds.mask.mean():.1%}")

b = next(iter(bundle.loader("train", 4, num_workers=0, mode="pretrain")))
for k, v in b.items():
    print(f"  {k:8s} {tuple(v.shape)}  {v.dtype}")

### L'augmentation tabulaire (corruption SCARF)

Sans augmentation du côté tabulaire, la perte contrastive s'effondre vers une solution triviale.
On remplace ~30 % des colonnes par la valeur d'une **autre ligne du train** : la vue reste
plausible colonne par colonne, mais les corrélations inter-colonnes sont cassées.

Détail qui compte : si la valeur donneuse était manquante, la vue reste manquante.

In [ ]:
from src.data.tabular import corrupt_tabular

i = 0
pre = bundle.pre
rng = np.random.default_rng(0)
xn0, xc0, mk0 = bundle.train_ds.x_num[i], bundle.train_ds.x_cat[i], bundle.train_ds.mask[i]
xn1, xc1, mk1 = corrupt_tabular(xn0, xc0, mk0, pre.marginal_num_, pre.marginal_cat_,
                                0.3, rng, marginal_mask=pre.marginal_mask_)

comp = pd.DataFrame({
    "feature":  s.feature_names,
    "original": [f"{v:.2f}" for v in xn0] + [str(v) for v in xc0],
    "corrompu": [f"{v:.2f}" for v in xn1] + [str(v) for v in xc1],
    "manq_orig": mk0, "manq_corr": mk1,
})
comp["changé"] = np.where(comp.original != comp.corrompu, "  <-- ", "")
comp

---
## Étape 3 — Les deux encodeurs

**Image** : backbone `timm`, `in_chans=1`, sans tête de classification.

**Tabulaire** : FT-Transformer *missing-aware*. Chaque colonne devient un token ; une colonne
manquante est remplacée par un embedding `[MISSING]` **appris et propre à cette colonne**.
S'y ajoutent un *field embedding* (identité de la colonne) et un embedding **sémantique** issu du
texte qui décrit la colonne.

In [ ]:
from src.data.semantic import build_semantic_matrix
from src.models.multimodal import build_pretrain_model

# Embeddings sémantiques : sentence-transformers si disponible, sinon hachage.
sem = build_semantic_matrix(s, backend="auto", verbose=True)
print("matrice sémantique :", sem.shape)

# IMPORTANT : on la sauvegarde et on l'inscrit dans la CONFIG. Tous les modèles
# construits ensuite (pré-entraînement ET fine-tuning) auront alors exactement
# la même architecture. Sinon le checkpoint SSL contient des poids
# `semantic_proj` que le modèle aval n'aurait pas, et le chargement échoue.
sem_path = Path(cfg.out_dir) / "semantic_emb.npy"
sem_path.parent.mkdir(parents=True, exist_ok=True)
np.save(sem_path, sem)
cfg["model"]["tabular"]["semantic_field_emb"] = str(sem_path)

# Les colonnes de sens voisin doivent être proches
S = sem @ sem.T
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(S, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(s.n_features)); ax.set_xticklabels(s.feature_names, rotation=90, fontsize=7)
ax.set_yticks(range(s.n_features)); ax.set_yticklabels(s.feature_names, fontsize=7)
ax.set_title("Similarité cosinus entre descriptions de colonnes")
plt.colorbar(im); plt.tight_layout(); plt.show()

In [ ]:
model = build_pretrain_model(cfg, s, sem).to(device)   # sem == celui de la config
print(f"encodeur image    : {count_params(model.image_encoder):>10,} paramètres")
print(f"encodeur tabulaire: {count_params(model.tabular_encoder):>10,} paramètres")

with torch.no_grad():
    out = model(b["image"].to(device), b["x_num"].to(device),
                b["x_cat"].to(device), b["mask"].to(device))
for k, v in out.items():
    print(f"  {k:10s} {tuple(v.shape)}")

### Vérification : le masque a-t-il vraiment un effet ?

Deux contrôles qui attrapent la majorité des bugs d'implémentation :

1. avec **100 % de valeurs manquantes**, la sortie doit être **finie** et **identique pour toutes
   les lignes** (il ne reste que des embeddings `[MISSING]`, donc plus aucune information) ;
2. la représentation doit **changer** quand on bascule le masque.

In [ ]:
model.eval()
with torch.no_grad():
    xn = b["x_num"].to(device); xc = b["x_cat"].to(device); mk = b["mask"].to(device)
    z_all_missing = model.encode_tabular(torch.zeros_like(xn), torch.zeros_like(xc),
                                         torch.ones_like(mk))
    z_none_missing = model.encode_tabular(xn, xc, torch.zeros_like(mk))

print("fini avec 100 % manquant     :", bool(torch.isfinite(z_all_missing).all()))
print("écart entre lignes (attendu 0):", f"{(z_all_missing - z_all_missing.mean(0)).abs().max():.2e}")
print("distance masqué / non masqué  :", f"{(z_all_missing[0] - z_none_missing[0]).norm():.3f}")
model.train();

---
## Étape 4 — Pré-entraînement contrastif

```
logits   = (z_I @ z_T.T) / τ
L_{I→T}  = CE(logits,   diag)
L_{T→I}  = CE(logits.T, diag)
L        = λ·L_{I→T} + (1-λ)·L_{T→I}
```

Deux indicateurs à surveiller pendant l'entraînement :

* la **perte** doit descendre nettement sous `log(B)` (la valeur du hasard) ;
* le **retrieval croisé** en validation (R@K) doit décoller au-dessus de `K/N`.

Si R@10 reste au niveau du hasard, inutile de continuer : l'alignement n'a pas eu lieu.

In [ ]:
from src.train.pretrain import pretrain

res_mm = pretrain(cfg, bundle=bundle, verbose=True,
                  mode="multimodal", tag="multimodal")
res_mm

In [ ]:
from src.eval.report import plot_pretrain_curves
from IPython.display import Image as IPImage, display

log = Path(cfg.out_dir) / "pretrain" / "multimodal" / "log.jsonl"
png = Path(cfg.out_dir) / "figures" / "pretrain_curves.png"
plot_pretrain_curves(log, png)
display(IPImage(str(png)))

hist = pd.DataFrame([__import__("json").loads(l) for l in log.read_text().splitlines() if l.strip()])
print(f"perte du hasard log(B) = {np.log(cfg.pretrain.batch_size):.3f}")
print(f"perte finale           = {hist['loss'].iloc[-1]:.3f}")
print(f"R@1  final (val)       = {hist['R@1_i2t'].iloc[-1]:.3f}  (hasard {hist['chance_R@1'].iloc[-1]:.4f})")
print(f"R@10 final (val)       = {hist['R@10_i2t'].iloc[-1]:.3f}")

### Visualiser l'espace latent partagé

Si l'alignement a fonctionné, chaque image et son dossier clinique tombent au même endroit.
Les traits gris relient les vraies paires : plus ils sont courts, mieux c'est.

In [ ]:
import torch.nn.functional as F
from src.eval.report import plot_embedding_umap
from src.utils import load_checkpoint

best = load_checkpoint(res_mm["best_ckpt"], map_location=device)
model.load_state_dict(best["model"]); model.eval()

Zi, Zt, Y = [], [], []
with torch.no_grad():
    for batch in bundle.loader("test", 128, shuffle=False, num_workers=0):
        o = model(batch["image"].to(device), batch["x_num"].to(device),
                  batch["x_cat"].to(device), batch["mask"].to(device))
        Zi.append(F.normalize(o["z_image"], dim=-1).cpu().numpy())
        Zt.append(F.normalize(o["z_tabular"], dim=-1).cpu().numpy())
        Y.append(batch["label"].numpy())
Zi, Zt, Y = np.concatenate(Zi), np.concatenate(Zt), np.concatenate(Y)

png = Path(cfg.out_dir) / "figures" / "latent_space.png"
plot_embedding_umap(Zi, Zt, Y, png, n=300)
display(IPImage(str(png)))

from src.eval.metrics import retrieval_metrics
r = retrieval_metrics(Zi, Zt, (1, 5, 10))
print({k: round(v, 4) for k, v in r.items()})

---
## Étape 5 & 6 — Fine-tuning et comparaison aux baselines

On jette les têtes de projection et on ne garde que les encodeurs.

Le protocole décisif est **`modalities="image"`** : au test, seul l'image est disponible.
On mesure donc ce que le pré-entraînement multimodal a *injecté* dans l'encodeur image.

Toutes les méthodes partagent le même `bundle` — même split, même prétraitement, même budget.

In [ ]:
from src.train.pretrain import pretrain as _pretrain
from src.train.finetune import finetune
from src.train.run_experiments import _cfg_with

# baseline SSL unimodale, pour isoler l'apport du tabulaire
res_sc = _pretrain(cfg, bundle=bundle, verbose=False,
                   mode="simclr_image", tag="simclr")

FRACTIONS = [0.01, 0.1, 1.0]
PLAN = [
    ("A_img_scratch",  None,                "image",   {"model__image__pretrained": False}),
    ("B_img_imagenet", None,                "image",   {"model__image__pretrained": True}),
    ("C_img_simclr",   res_sc["best_ckpt"], "image",   {}),
    ("D_tab_only",     None,                "tabular", {}),
    ("E_concat_naive", None,                "both",
     {"model__tabular__arch": "naive_mlp",
      "model__tabular__semantic_field_emb": None}),
    ("F_ours_img",     res_mm["best_ckpt"], "image",   {}),
    ("G_ours_both",    res_mm["best_ckpt"], "both",    {}),
]

rows = []
for frac in FRACTIONS:
    for tag, ckpt, mods, ov in PLAN:
        c = _cfg_with(cfg, **ov) if ov else cfg
        r = finetune(c, bundle=bundle, pretrained_ckpt=ckpt, modalities=mods,
                     label_fraction=frac, tag=f"{tag}_f{frac}", verbose=False)
        r["method"] = tag
        rows.append(r)
        print(f"{tag:16s} @{frac:>5.0%}  AUROC={r['test_auroc']:.4f} "
              f"[{r['test_auroc_lo']:.3f}-{r['test_auroc_hi']:.3f}]")

results = pd.DataFrame(rows)
results.to_csv(Path(cfg.out_dir) / "results.csv", index=False)

In [ ]:
piv = results.pivot_table(index="method", columns="label_fraction", values="test_auroc")
piv.style.background_gradient(cmap="Greens", axis=0).format("{:.4f}")

In [ ]:
from src.eval.report import plot_label_efficiency

png = Path(cfg.out_dir) / "figures" / "label_efficiency.png"
plot_label_efficiency(Path(cfg.out_dir) / "results.csv", png)
display(IPImage(str(png)))

### Lire le tableau sans se raconter d'histoires

Les deux seules comparaisons qui portent une conclusion :

* **F − C** — apport du **tabulaire** dans le SSL (les deux sont du SSL image-only au test) ;
* **F − E** — apport du **SSL** par rapport à une fusion supervisée naïve.

Et la règle de prudence : si les intervalles de confiance bootstrap se chevauchent largement,
on ne conclut pas.

In [ ]:
for frac, g in results.groupby("label_fraction"):
    g = g.set_index("method")
    if "F_ours_img" not in g.index:
        continue
    f = g.loc["F_ours_img"]
    print(f"\n--- {frac:.0%} des étiquettes ---")
    print(f"  F_ours_img : {f.test_auroc:.4f}  IC95 [{f.test_auroc_lo:.3f}, {f.test_auroc_hi:.3f}]")
    for base in ("C_img_simclr", "E_concat_naive", "A_img_scratch"):
        if base in g.index:
            bb = g.loc[base]
            delta = f.test_auroc - bb.test_auroc
            overlap = not (f.test_auroc_lo > bb.test_auroc_hi or bb.test_auroc_lo > f.test_auroc_hi)
            verdict = "IC qui se chevauchent -> non concluant" if overlap else "écart significatif"
            print(f"  vs {base:16s} {delta:+.4f}   ({verdict})")

---
## Étape 7 — Ablation : robustesse aux métadonnées manquantes

Situation réelle : à l'inférence, le dossier clinique est souvent incomplet, voire absent.
On prend **le même modèle fine-tuné** en image+tabulaire et on masque progressivement les
métadonnées au test.

L'argument défendable est là : la courbe doit chuter **moins vite** que pour une concaténation
naïve, parce que l'encodeur image a déjà absorbé une partie de l'information tabulaire pendant
le pré-entraînement contrastif.

In [ ]:
from src.train.run_experiments import missingness_robustness

rob, base_r = missingness_robustness(cfg, bundle, res_mm["best_ckpt"],
                                     rates=(0.0, 0.25, 0.5, 0.75, 1.0), verbose=False)
display(rob.round(4))

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(rob.extra_missing_rate * 100, rob.test_auroc, marker="o", lw=2.2,
        label="SSL multimodal (image+tab)")
ax.axhline(results.query("method=='F_ours_img' and label_fraction==1.0").test_auroc.iloc[0],
           ls="--", c="gray", label="SSL multimodal, image seule au test")
ax.set_xlabel("métadonnées masquées au test (%)"); ax.set_ylabel("AUROC test")
ax.set_title("Dégradation quand le dossier clinique se vide")
ax.grid(alpha=.3); ax.legend(); plt.tight_layout(); plt.show()

---
## Bilan

Ce qui a été démontré, et sur quelle preuve :

| Affirmation | Preuve dans ce notebook |
|---|---|
| L'alignement image ↔ tabulaire s'apprend réellement | R@10 très au-dessus du hasard, espace latent superposé |
| Le SSL multimodal bat le SSL unimodal | écart F − C, avec IC bootstrap |
| Il bat aussi la fusion supervisée naïve | écart F − E |
| Le gain est maximal en régime peu annoté | courbe d'efficacité en labels |
| L'encodeur gère les manquants sans imputation | sortie finie et constante à 100 % de manquants ; courbe de robustesse |

### Pour aller plus loin

1. Plusieurs graines, report en moyenne ± écart-type.
2. Passer à **CBIS-DDSM** (`data.source=cbis_ddsm`) — manquance structurelle réelle.
3. Reconstruction tabulaire depuis l'image en tâche auxiliaire.
4. Attention croisée pixels ↔ colonnes, pour l'interprétabilité clinique.
5. Transfert CBIS-DDSM → VinDr-Mammo : c'est là que les embeddings sémantiques de colonnes
   deviennent indispensables, et probablement la contribution la plus originale accessible.

Voir `README.md` §9.
